### Alura_formato_arquivo

# JSON

**Lendo um arquivo JSON**

In [0]:
df = spark.read.json("/FileStore/tables/arquivos_curso/PNSB.json")
display(df)

In [0]:
## Renomear colunas
df = df.withColumnRenamed("D1C","cod_regiao") \
            .withColumnRenamed("D1N", "regiao")\
            .withColumnRenamed("D2C","cod_variavel") \
            .withColumnRenamed("D2N", "variavel") \
            .withColumnRenamed("D3C", "cod_ano") \
            .withColumnRenamed("D3N", "ano") \
            .withColumnRenamed("D4C","cod_doenca") \
            .withColumnRenamed("D4N", "doenca") \
            .withColumnRenamed("MC","cod_medida") \
            .withColumnRenamed("MN", "medida") \
            .withColumnRenamed("NC","cod_nivel_territorial") \
            .withColumnRenamed("NN", "nivel territorial") \
            .withColumnRenamed("V","valor")

display (df)

In [0]:
## Remover a primeira linha
df = df.filter(df.valor!='Valor')
display(df)

In [0]:
## Visualizar a tipagem das colunas
df.printSchema()

In [0]:
## modificar o tipo das colunas (cast)
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType

df_new = df.withColumn ("cod_regiao", col("cod_regiao").cast (IntegerType())) \ 
                    .withColumn("cod_variavel", col("cod_variavel").cast (IntegerType())) \ 
                    .withColumn("cod_ano", col("cod_ano").cast (IntegerType())) \ 
                    .withColumn("ano", col("ano").cast (IntegerType())) \
                    .withColumn("cod_doenca", col("cod_doenca").cast (IntegerType())) \
                    .withColumn("cod_medida", col("cod_medida").cast (IntegerType())) \
                    .withColumn("cod_nivel_territorial", col("cod_nivel_territorial").cast (IntegerType())) \
                    .withColumn("valor", col("valor").cast (IntegerType()))

In [0]:
## Visualizar a tipagem das colunas
df.printSchema()

**Salvando o JSON comprimido**

In [0]:
#Verificar o conteúdo da pasta
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso"))

In [0]:
## reduzir o tamanho do arquivo
df_new.write \
    .option("compression", "gzip") \
    .mode("overwrite") \
    .format("json") \
    .save("/FileStore/tables/arquivos_curso/json_gzip")

In [0]:
#Verificar o conteúdo da pasta
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/json_gzip"))

In [0]:
## Fazer a leitura do arquivo comprimido
df = spark.read \
        .option("compression", "gzip") \
        .json("/FileStore/tables/arquivos_curso/json_gzip/")
        
display(df)

In [0]:
## salvar o dataframe em outro formato
df.write \
    .option("sep", ",") \
    .format("csv") \
    .save("/FileStore/tables/arquivos_curso/pnsb_csv/")

In [0]:
#Verificar o conteúdo da pasta
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/pnsb_csv"))

# CSV

_formas de realizar a leitura do arquivo CSV:_

`df_csv = spark.read \
          .csv("<caminho_do_arquivo>")`

ou

`df_csv = spark.read\
  .format("csv") \
  .load("<caminho_do_arquivo>")`


In [0]:
## Lendo um arquivo CSV
df_csv = spark.read.csv('/FileStore/tables/arquivos_curso/pnsb_csv')
display(df_csv)

In [0]:
##salvar o arquivo novamente com especificação de cabeçalho
df.write \
    .option("sep", ",") \
    .option("header", True) \
    .mode("overwrite") \
    .format("csv") \
    .save("/FileStore/tables/arquivos_curso/pnsb_csv/")

In [0]:
## Lendo um arquivo CSV
df_csv = spark.read.csv('/FileStore/tables/arquivos_curso/pnsb_csv')
display(df_csv)

In [0]:
##especificar a primeira linha como cabeçalho
df_csv = spark.read.csv('/FileStore/tables/arquivos_curso/pnsb_csv', header=True)
display(df_csv)

In [0]:
##verificar a tipagem das colunas
df_csv.printSchema()

In [0]:
## modificar a tipagem das colunas
df_csv = spark.read.csv('/FileStore/tables/arquivos_curso/pnsb_csv', header=True, inferSchema=True)
display(df_csv)

In [0]:
##verificar a tipagem das colunas
df_csv.printSchema()

**Compressão do arquivo
**

In [0]:
## Escrevendo arquivo CSV com compressão
df_csv.write \
            .option("compression", "gzip") \
            .option("header", "true") \
            .option("sep", ",") \
            .format("csv") \
            .save("/FileStore/tables/arquivos_curso/csv_gzip/")

In [0]:
## visualizar o conteudo da pasta
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/csv_gzip"))

In [0]:
## vamos verificar o tamanho do arquivo anterior executando o comando a seguir:
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/pnsb_csv"))

In [0]:
## Leitura do arquivo
df = spark.read \
            .option("compression", "gzip") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .option("sep", ",") \
            .csv("/FileStore/tables/arquivos_curso/csv_gzip")

display(df)

# TXT

**Salvando o DataFrame em txt**

In [0]:
#preencher os dados nulos com um valor constante
df = df.na.fill(value=0, subset=["valor"])
display(df)

In [0]:
#juntar as colunas
from pyspark.sql.functions import concat_ws

df_uma_coluna = df.select(concat_ws("|", *df.columns).alias('dados'))
display(df_uma_coluna)

In [0]:
#Escrever o dataframe
df_uma_coluna.write \
            .format("text") \
            .mode("overwrite") \
            .save("/FileStore/tables/arquivos_curso/txt/")

In [0]:
#visualizar o conteúdo da pasta
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/txt/"))

In [0]:
#leitura do arquivo
df_text = spark.read.format("text") \
            .load("/FileStore/tables/arquivos_curso/txt/")
            
display(df_text)

In [0]:
#separar colunas
df_text = spark.read \
        .option("header", "false") \
        .option("delimiter", "|") \
        .option("inferSchema", "true") \
        .format("csv") \
        .load("/FileStore/tables/arquivos_curso/txt/")

display(df_text)

In [0]:
#renomear as colunas
df_text_2 = df_text.withColumnRenamed ("_c0", "ano") \
                .withColumnRenamed("_c1","cod_ano") \
                .withColumnRenamed("_c2","cod_doenca") \ 
                .withColumnRenamed("_c3","cod_medida") \
                .withColumnRenamed("_c4","cod_nivel_territorial") \
                .withColumnRenamed("_c5","cod_regiao")\
                .withColumnRenamed("_c6", "cod_variavel") \
                .withColumnRenamed("_c7", "doenca") \
                .withColumnRenamed("_c8", "medida") \
                .withColumnRenamed("_c9", "nivel_territorial") \
                .withColumnRenamed("_c10", "regiao") \ 
                .withColumnRenamed("_c11","valor") \
                .withColumnRenamed("_c12","variavel")

display (df_text_2)

**Salvando o arquivo txt comprimido**

In [0]:
df_text_2.write \
    .mode("overwrite") \
    .option("compression", "gzip") \
    .format("text") \
    .save("/FileStore/tables/arquivos_curso/txt_gzip/")

In [0]:
df_uma_coluna.write \
    .mode("overwrite") \
    .option("compression", "gzip") \
    .format("text") \
    .save("dbfs:/FileStore/tables/arquivos_curso/txt_gzip/")

In [0]:
#Verificando se foi salvo corretamente
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/txt_gzip/"))

In [0]:
#comparando o ganho de compressão com o arquivo anterior
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/txt/"))

In [0]:
#fazer a leitura do arquivo
df = spark.read \
     .option("compression", "gzip") \
   .option("inferSchema", "true") \
   .option("sep", "|") \
   .csv("/FileStore/tables/arquivos_curso/txt_gzip/")

display(df)

In [0]:
#Renomear colunas
df_renomeado = df_text.withColumnRenamed ("_c0", "ano") \
  .withColumnRenamed("_c1","cod_ano") \
  .withColumnRenamed("_c2","cod_doenca") \ 
  .withColumnRenamed("_c3","cod_medida") \
  .withColumnRenamed("_c4","cod_nivel_territorial") \
  .withColumnRenamed("_c5","cod_regiao")\
  .withColumnRenamed("_c6", "cod_variavel") \
  .withColumnRenamed("_c7", "doenca") \
  .withColumnRenamed("_c8", "medida") \
  .withColumnRenamed("_c9", "nivel_territorial") \
  .withColumnRenamed("_c10", "regiao") \ 
  .withColumnRenamed("_c11","valor") \
  .withColumnRenamed("_c12","variavel")

display (df_renomeado)

In [0]:
#Salvando o arquivo em outro formato
df_renomeado.write \
    .mode("overwrite") \
    .format('avro') \
    .save("/FileStore/tables/arquivos_curso/avro/")

In [0]:
#Verificar se salvou corretamente
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/avro/"))

#AVRO

**Lendo um arquivo no formato AVRO**

In [0]:
df_avro = spark.read \
    .format("avro") \
    .load("/FileStore/tables/arquivos_curso/avro/")

display(df_avro)

In [0]:
#Lendo apenas os arquivos avro da pasta
df_avro = spark.read \
    .format("avro") \
    .load("/FileStore/tables/arquivos_curso/avro/", pathGlobFilter="*.avro")

display(df_avro)

In [0]:
#Escrevendo o arquivo AVRO com compressão
df_avro.write \
    .mode("overwrite") \
    .option("compression", "deflate") \
    .format('avro') \
    .save("/FileStore/tables/arquivos_curso/avro_deflate")


In [0]:
#Validar se for salvo corretamente
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/avro_deflate"))

In [0]:
#Validar o tamanho da compressão padrão
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/avro"))

In [0]:
#Tornar um tipo de compressão padrão
spark.conf.set("spark.sql.avro.compression.codec", "deflate")

In [0]:
#faremos novamente a escrita, mas sem especificar o codec
df_avro.write \
    .mode("overwrite") \
    .format('avro') \
    .save("/FileStore/tables/arquivos_curso/avro_deflate2")

In [0]:
#Validar se foi salvo corretamente sem especificar o codec
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/avro_deflate2"))

In [0]:
#Especificar o nivel de compressão (padrão é 6)
spark.conf.set("spark.sql.avro.deflate.level", "8")

In [0]:
#salvar sobrescrevendo o arquivo anterior
df_avro.write \
    .mode("overwrite") \
    .format('avro') \
    .save("/FileStore/tables/arquivos_curso/avro_deflate2")

In [0]:
#validando o conteúdo da pasta
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/avro_deflate2"))

# PARQUET

**Lendo e escrevendo arquivos PARQUET**

In [0]:
df_avro.write \
        .mode("overwrite") \
        .format('parquet') \
        .save("/FileStore/tables/arquivos_curso/parquet")

In [0]:
#Verificar se foi salvo corretamente
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/parquet"))

In [0]:
#Salvar o arquivo com compressão gzip
df_avro.write \
        .mode("overwrite") \
        .option("compression", "gzip") \
        .format('parquet') \
        .save("/FileStore/tables/arquivos_curso/parquet_gzip")

In [0]:
#Validar se foi salvo corretamente
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/parquet_gzip"))

In [0]:
#ler o arquivo
df_parquet = spark.read.format("parquet") \
    .load("/FileStore/tables/arquivos_curso/parquet_gzip", compression='gzip')

display(df_parquet)

### Particionamento
`df.write\
.partitionBy ("coluna_para_particionar")\
.format("formato_do_arquivo")\
.save("caminho/para/salvar")`

In [0]:
#Verificar quantos valores distintos a coluna tem, para definir a coluna para o particionamento
df_parquet.select("cod_doenca").distinct().show()

In [0]:
#Verificar quantos valores distintos a coluna tem, para definir a coluna para o particionamento
df_parquet.select("nivel_territorial").distinct().show()

In [0]:
#particionando o dataframe
df_parquet.write \
    .partitionBy("nivel_territorial") \
    .mode("overwrite") \
    .parquet("/FileStore/tables/arquivos_curso/parquet_particionado")

In [0]:
#validar o esquema de pastas desses arquivos
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/parquet_particionado"))

In [0]:
#Abrir uma pasta para analisar
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/parquet_particionado/nivel_territorial=Brasil"))

In [0]:
# particionar por mais de uma coluna
df_parquet.write \
    .partitionBy("nivel_territorial", "cod_doenca) \
    .mode("overwrite") \
    .parquet("/FileStore/tables/arquivos_curso/parquet_multi_particionado")

In [0]:
#validar esquema de pastas
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/parquet_multi_particionado"))

In [0]:
#Analisar uma pasta para validar o conteúdo
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/parquet_multi_particionado/nivel_territorial=Grande Região"))

In [0]:
#fazendo a leitura de uma partição
df_120943 = spark.read \
    .parquet(/'FileStore/tables/arquivos_curso/parquet_multi_particionado/nivel_territorial=Grande Região/cod_doenca=120943/')

display(df_120943)

In [0]:
#Voltando para o dataframe original
df = spark.read\
    .parquet('FileStore/tables/arquivos_curso/parquet_particionado')

display(df)

### ORC

**Escrevendo e lendo arquivos ORC**

In [0]:
# Verificar a escrita do dataframe no formato ORC
df.write \
    .format('orc') \
    .save("/FileStore/tables/arquivos_curso/orc")

In [0]:
# verificar o conteúdo da pasta
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/orc"))

In [0]:
#Comprimir o dataframe
df.write.format("orc") \
    .mode("overwrite") \
    .option("compression", "zlib") \
    .save("/FileStore/tables/arquivos_curso/orc_zlib"

In [0]:
#verificar a estrutura da pasta
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/orc_zlib"))

In [0]:
#Salvando no formato ORC
df_orc = spark.read\
    .option("compression", "zlib") \
    .format("orc") \
    .load("/FileStore/tables/arquivos_curso/orc_zlib"

display(df_orc)

### Agrupando as partições criadas

In [0]:
#O método .coalesce(), do PySpark, para diminuir o número de partições
df_orc.coalesce(1)\
    .write \
    .format("orc") \
    .mode("overwrite") \
    .save("/FileStore/tables/arquivos_curso/orc_junto_snappy")

In [0]:
#verificar a pasta
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/orc_junto_snappy/"))

In [0]:
#Para fins de comparação, faremos este mesmo processo com a compressão zlib
df_orc.coalesce(1)\
    .write \
    .format("orc") \
    .mode("overwrite") \
    .option("compression", "zlib") \
    .save("/FileStore/tables/arquivos_curso/orc_junto_zlib")

In [0]:
#Verificar pasta
display(dbutils.fs.ls("/FileStore/tables/arquivos_curso/orc_junto_zlib"))

In [0]:
# Leitura do arquivo ORC comprimido
df_orc_zlib = spark.read \
    .option("compression", "zlib") \
    .orc("/FileStore/tables/arquivos_curso/orc_junto_zlib")

display(df_orc_zlib)